# Inference Visualizer
## Developed by Moose Abou-Harb on behalf of Paccar Inc

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

import json
import pathlib
import math

store_folder_path = pathlib.Path.cwd().parent / "test_data"

earth_radius = 6378137.0
deg_to_rad = math.pi / 180

modalities = ["camera", "lidar", "radar", "gt", "fusion"]

def main():
    #Load up the data from the sim
    gt_data = {}
    inferences = {}
    try:
        for modality in modalities:
            modality_path = store_folder_path / f"{modality}_sim_results.json"
            mod_data = try_load_json(modality_path)
            if modality == "gt":
                gt_data = mod_data
            inferences[modality] = mod_data["inferences"]
    except Exception as e:
        print("Failed to load sim data!")
        return

    #Get the origin
    origin = gt_data["start_pos"]

    #Shove that data into a dataframe
    main_df = pd.DataFrame()
    for modality in modalities:
        new_df = pd.DataFrame(inferences[modality])
        new_df["modality"] = modality
        main_df = pd.concat([main_df, new_df])

    #Expand lat/long/alt into local coords
    main_df[["x", "y", "z"]] = main_df.apply(
        lambda row : geo_to_local(origin, [row["latitude"], row["longitude"], row["altitude"]]), 
        axis=1,
        result_type="expand"
    )

    #Create an origin dataframe
    origin_df = pd.DataFrame([{
        "timestamp" : 0,
        "class" : "origin",
        "latitude" : origin[0],
        "longitude" : origin[1],
        "altitude" : origin[2],
        "dimensions" : [0, 0, 0],
        "obj_id" : -1,
        "modality" : "origin",
        "x" : 0,
        "y" : 0,
        "z" : 0
    }])

    #Merge in the origin point
    main_df = pd.concat([origin_df, main_df])

    #Expand dimensions into individual columns
    main_df[["dx", "dy", "dz"]] = pd.DataFrame(main_df["dimensions"].tolist(), index=main_df.index)
    main_df = main_df.drop("dimensions", axis=1)
    main_df = main_df.sort_values(by="obj_id")
    
    display(main_df.head(30))

    #Old graph display code
    # fig = px.scatter_3d(main_df, x="x", y="y", z="z", opacity=0.7, color="modality")
    # fig.update_traces(marker=dict(size=5))
    # fig.show()

    color_map = {
        "camera" : "blue",
        "lidar" : "yellow",
        "radar" : "green",
        "fusion" : "black",
        "gt" : "white"
    }

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode="markers",
        marker=dict(size=5, color="black")
    ))
    for _, row in main_df.iterrows():
        color = color_map.get(row["modality"], "gray")
        fig.add_trace(
            wireframe_box(
                center=(row["x"], row["y"], row["z"]),
                dimensions=[row["dx"], row["dy"], row["dz"]],
                color=color
            )
        )
    fig.update_layout(
        scene = dict(
            aspectmode = "data",
            xaxis_title = "x",
            yaxis_title = "y",
            zaxis_title = "z"
        )
    )

    fig.show()

def try_load_json(fp: str) -> dict:
    try:
        with open(fp, "r") as file:
            data = json.load(file)
            return data
    except FileNotFoundError:
        print(f"Failed to load file: {fp}")
    except json.JSONDecodeError:
        print(f"Selected file contains illegal JSON: {fp}")
    except Exception as e:
        print(f"Something went wrong while loading: {fp}, {e}")

def geo_to_local(origin, target):
    d_lat = target[0] - origin[0]
    d_lon = target[1] - origin[1]
    d_alt = target[2] - origin[2]
    y_meters = d_lat * deg_to_rad * earth_radius

    radius_at_lat = earth_radius * math.cos(origin[0] * deg_to_rad)
    x_meters = d_lon * deg_to_rad * radius_at_lat

    return [x_meters, y_meters, d_alt]

def wireframe_box(center, dimensions, color="blue"):
    x, y, z = center
    dx, dy, dz = dimensions

    hx, hy, hz = dx / 2, dy / 2, dz / 2

    vertices = [
        (x-hx, y-hy, z-hz),
        (x+hx, y-hy, z-hz),
        (x+hx, y+hy, z-hz),
        (x-hx, y+hy, z-hz),
        (x-hx, y-hy, z+hz),
        (x+hx, y-hy, z+hz),
        (x+hx, y+hy, z+hz),
        (x-hx, y+hy, z+hz),
    ]

    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7)
    ]

    xs, ys, zs = [], [], []
    for i, j in edges:
        xs += [vertices[i][0], vertices[j][0], None]
        ys += [vertices[i][1], vertices[j][1], None]
        zs += [vertices[i][2], vertices[j][2], None]

    return go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False
    )

if __name__ == "__main__":
    main()


,timestamp,class,latitude,longitude,altitude,obj_id,modality,x,y,z,dx,dy,dz
0,0.000000,origin,57.097690,-8.803518,2192.387221,-1.0,origin,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
0,16.946319,vehicle,57.097652,-8.803521,2195.702196,0.0,camera,-0.140711,-4.203269,3.314975,0.454299,0.789583,0.988481
0,16.977611,vehicle,57.097651,-8.803525,2195.632236,0.0,lidar,-0.430140,-4.269984,3.245015,0.437497,0.739932,1.005885
0,16.998764,traffic_cone,57.097653,-8.803528,2195.728439,0.0,radar,-0.615606,-4.096612,3.341218,0.505385,0.882474,0.876966
4,0.000000,FixMe,57.097626,-8.803835,2193.705942,0.0,fusion,-19.187745,-7.024964,1.318721,0.745481,-0.016403,1.020990
5,0.000000,FixMe,57.097631,-8.803838,2193.660410,0.0,fusion,-19.360531,-6.547421,1.273189,0.743398,0.158287,0.875331
6,0.000000,FixMe,57.097629,-8.803839,2193.641293,0.0,fusion,-19.419867,-6.690288,1.254072,0.731517,0.111164,0.915435
7,0.000000,FixMe,57.097652,-8.803525,2195.687624,0.0,fusion,-0.395486,-4.189955,3.300403,0.465727,0.803997,0.957111
0,0.000000,FixMe,57.097594,-8.803217,2187.919432,0.0,fusion,18.239061,-10.662818,-4.467790,0.183683,0.186956,0.945322
1,0.000000,FixMe,57.097596,-8.803224,2187.867461,0.0,fusion,17.789901,-10.452888,-4.519760,0.270360,0.342925,0.845274
